# Critical Input DEQN: Repair-Aware Taylor Rule

This notebook trains the adaptation-aware Taylor rule, `policy="repair_aware"`. The rule starts from the fixed-intercept Taylor rate, applies the same cap-pressure bottleneck adjustment as notebook 10, and then adds an extra repair-support adjustment when the repair value is close to the private investment threshold.

In [ ]:
# Configure paths and repair-aware Taylor settings.
from pathlib import Path
import json
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'

def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError('No checkpoint found:\n' + '\n'.join(str(p) for p in paths))

NATURAL_CKPT = first_existing([
    ARTIFACT_ROOT / 'natural' / 'checkpoints' / 'natural_best.pt',
    ARTIFACT_ROOT / 'natural' / 'natural.pt',
])
OUT = ARTIFACT_ROOT / 'repair_aware_taylor'
OUT.mkdir(parents=True, exist_ok=True)

RULE_STEPS = 8_000
QMC_TRAIN = 256
QMC_VAL = 512
NATURAL_ORACLE_NODES = 32
NATURAL_ORACLE_CHUNK_SIZE = 8192
N_VAL_STATES = 1024
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
LOG_EVERY = 100
BATCH_SIZE = 2048
SIM_BATCH_SIZE = 512
EPISODE_LENGTH = 20
EPISODE_UPDATES_PER_EPISODE = 2
EPISODE_BROAD_SHARE = 0.50
CHECKPOINT_EVERY = 1000
STOP_VAL_STATES = 512
SCENARIO_Q_WEIGHT = 25.0
CALM_ANCHOR_WEIGHT = 5.0
CALM_RESIDUAL_WEIGHT = 5.0
SCENARIO_BURNIN = 5
SCENARIO_HORIZON = 10
SCENARIO_LOSS_INTERVAL = 25
TARGET_SCENARIO_Q_RMS = 1e-2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'

print('ROOT:', ROOT)
print('natural:', NATURAL_CKPT)
print('output:', OUT)
print('device:', DEVICE)

def run_stream(cmd, cwd=ROOT):
    print('Running:', ' '.join(map(str, cmd)))
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
# Train the repair-aware Taylor network.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_train',
    '--output-dir', str(OUT),
    '--policies', 'repair_aware',
    '--natural-checkpoint', str(NATURAL_CKPT),
    '--natural-benchmark', 'oracle',
    '--natural-oracle-nodes', str(NATURAL_ORACLE_NODES),
    '--natural-oracle-chunk-size', str(NATURAL_ORACLE_CHUNK_SIZE),
    '--rule-steps', str(RULE_STEPS),
    '--rule-trainer', 'episode',
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
    '--batch-size', str(BATCH_SIZE),
    '--sim-batch-size', str(SIM_BATCH_SIZE),
    '--episode-length', str(EPISODE_LENGTH),
    '--episode-updates-per-episode', str(EPISODE_UPDATES_PER_EPISODE),
    '--episode-broad-share', str(EPISODE_BROAD_SHARE),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--rule-scenario-q-weight', str(SCENARIO_Q_WEIGHT),
    '--rule-calm-anchor-weight', str(CALM_ANCHOR_WEIGHT),
    '--rule-calm-residual-weight', str(CALM_RESIDUAL_WEIGHT),
    '--rule-scenario-burnin', str(SCENARIO_BURNIN),
    '--rule-scenario-horizon', str(SCENARIO_HORIZON),
    '--rule-scenario-loss-interval', str(SCENARIO_LOSS_INTERVAL),
    '--target-scenario-q-rms', str(TARGET_SCENARIO_Q_RMS),
]
run_stream(cmd)


In [ ]:
# Inspect out-of-sample residual diagnostics for repair-aware Taylor.
eval_path = OUT / 'repair_aware_eval.json'
if not eval_path.exists():
    raise FileNotFoundError(f'Missing eval file. If training was interrupted, use {OUT / "checkpoints" / "repair_aware_best.pt"} in a diagnostics cell.')
with eval_path.open('r', encoding='utf-8') as fh:
    repair_aware_eval = json.load(fh)
repair_aware_eval


In [ ]:
# Mechanism diagnostic: does the repair-aware rule actually move the policy rate?
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.critical_input_deqn.config import BaselineParams, NetworkConfig, QMCConfig
from src.critical_input_deqn.experiments import params_from_metadata, resolve_params
from src.critical_input_deqn.natural_oracle import NaturalOracleNet
from src.critical_input_deqn.train import make_rule_net, load_checkpoint
from src.critical_input_deqn.postprocess import simulate_rule_ir_scenarios, evaluate_rule_path

def latest_step_checkpoint(folder, name):
    files = sorted((Path(folder) / 'checkpoints').glob(f'{name}_step_*.pt'))
    return files[-1] if files else None

ra_step_ckpt = latest_step_checkpoint(OUT, 'repair_aware')
ra_ckpt_candidates = [
    OUT / 'checkpoints' / 'repair_aware_best.pt',
    OUT / 'repair_aware.pt',
]
if ra_step_ckpt is not None:
    ra_ckpt_candidates.append(ra_step_ckpt)
RA_CKPT = first_existing(ra_ckpt_candidates)
RUN_CONFIG_PATH = OUT / 'run_config.json'
if RUN_CONFIG_PATH.exists():
    with RUN_CONFIG_PATH.open('r', encoding='utf-8') as fh:
        run_config = json.load(fh)
    params = params_from_metadata({'config': run_config}, fallback=BaselineParams())
else:
    payload = torch.load(RA_CKPT, map_location=DEVICE)
    ckpt_meta = payload.get('metadata', {}) if isinstance(payload, dict) else {}
    run_config = ckpt_meta.get('config', {}) if isinstance(ckpt_meta, dict) else {}
    params, _ = resolve_params('baseline', None)
    params = params_from_metadata({'config': run_config}, fallback=params)
net_data = run_config.get('network', {})
net_cfg = NetworkConfig(
    hidden_width=int(net_data.get('hidden_width', HIDDEN_WIDTH)),
    hidden_depth=int(net_data.get('hidden_depth', HIDDEN_DEPTH)),
    activation=str(net_data.get('activation', 'selu')),
    init_scale=float(net_data.get('init_scale', 0.01)),
)
torch_dtype = torch.float64 if DTYPE == 'float64' else torch.float32

natural_net = NaturalOracleNet(
    params=params,
    qmc_cfg=QMCConfig(n_train=NATURAL_ORACLE_NODES, seed=991),
    n_nodes=NATURAL_ORACLE_NODES,
    device=DEVICE,
    dtype=torch_dtype,
    chunk_size=NATURAL_ORACLE_CHUNK_SIZE,
)
natural_net.eval()

ra_net = make_rule_net(net_cfg, device=DEVICE, dtype=torch_dtype)
load_checkpoint(RA_CKPT, ra_net, map_location=DEVICE)
ra_net.eval()

DIAG_IR_BURNIN = 400
DIAG_IR_HORIZON = 160
DIAG_IR_PRESTEPS = 5
DIAG_IR_RELIEF_LAG = 8

labels, ir_states = simulate_rule_ir_scenarios(
    policy='repair_aware',
    rule_net=ra_net,
    natural_net=natural_net,
    params=params,
    burnin=DIAG_IR_BURNIN,
    horizon=DIAG_IR_HORIZON,
    presteps=DIAG_IR_PRESTEPS,
    relief_lag=DIAG_IR_RELIEF_LAG,
    device=DEVICE,
    dtype=torch_dtype,
)
_, ir_defs = evaluate_rule_path(
    ir_states,
    policy='repair_aware',
    rule_net=ra_net,
    natural_net=natural_net,
    params=params,
)
ir_defs['R_over_R_standard'] = ir_defs['R'] / np.clip(ir_defs['R_standard'], 1e-12, None)

print('repair-aware checkpoint:', RA_CKPT)
print('natural_benchmark: oracle')
print('labels:', labels)
print('repair_margin_trigger:', params.repair_margin_trigger)
print('repair_cap_pressure_trigger:', params.repair_cap_pressure_trigger)
print('phi_repair:', params.phi_repair)

mechanism_vars = [
    'R_standard', 'R', 'R_over_R_standard',
    'repair_margin_standard', 'repair_support', 'repair_adjustment',
    'cap_pressure_policy', 'Q_A', 'I_A', 'A',
]
rows = []
event_idx = int(DIAG_IR_PRESTEPS)
for j, label in enumerate(labels):
    if label == 'no_event':
        continue
    for var in mechanism_vars:
        if var not in ir_defs:
            continue
        series = np.asarray(ir_defs[var])[:, j]
        base = np.asarray(ir_defs[var])[:, 0]
        rows.append({
            'scenario': label,
            'variable': var,
            'event': float(series[event_idx]),
            'min': float(np.nanmin(series)),
            'max': float(np.nanmax(series)),
            'min_dev_from_no_event': float(np.nanmin(series - base)),
            'max_dev_from_no_event': float(np.nanmax(series - base)),
        })
mechanism_summary = pd.DataFrame(rows)
display(mechanism_summary)

for label in ['D_1x', 'D_3x', 'D_1x_X_lag', 'D_3x_X_lag']:
    if label not in labels:
        continue
    j = labels.index(label)
    print(f"\n{label} event mechanism:")
    for var in mechanism_vars:
        if var in ir_defs:
            print(f"  {var:28s} {float(np.asarray(ir_defs[var])[event_idx, j]): .6e}")

plot_vars = [
    'R_standard', 'R', 'R_over_R_standard',
    'repair_margin_standard', 'repair_support', 'repair_adjustment',
    'cap_pressure_policy', 'Q_A', 'I_A', 'A',
]
t = np.arange(np.asarray(ir_defs[plot_vars[0]]).shape[0]) - DIAG_IR_PRESTEPS
fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=True)
axes = axes.reshape(-1)
for ax, var in zip(axes, plot_vars):
    for j, label in enumerate(labels):
        if label == 'no_event':
            continue
        ax.plot(t, np.asarray(ir_defs[var])[:, j], label=label)
    ax.axvline(0, color='0.55', linewidth=0.8)
    if var in {'R_over_R_standard', 'repair_adjustment'}:
        ax.axhline(1.0, color='0.35', linestyle='--', linewidth=0.8)
    if var == 'repair_margin_standard':
        ax.axhline(float(params.repair_margin_trigger), color='tab:red', linestyle='--', linewidth=0.8)
    if var == 'cap_pressure_policy':
        ax.axhline(float(params.repair_cap_pressure_trigger), color='tab:red', linestyle='--', linewidth=0.8)
    ax.set_title(var)
    ax.grid(True, alpha=0.25)
axes[-1].legend(loc='best')
fig.tight_layout()


In [ ]:
# Optional: compare rule-policy validation diagnostics if the other Taylor rules already exist.
import pandas as pd

paths = {
    'fixed': ARTIFACT_ROOT / 'fixed_taylor' / 'fixed_eval.json',
    'natural_rate_adjusted': ARTIFACT_ROOT / 'modified_taylor' / 'ba_eval.json',
    'bottleneck': ARTIFACT_ROOT / 'bottleneck_taylor' / 'bottleneck_eval.json',
    'repair_aware': OUT / 'repair_aware_eval.json',
}
keys = [
    'rms', 'max_abs', 'hh_euler.rms', 'resource.rms', 'price_index.rms',
    'calvo_S.rms', 'calvo_F.rms', 'Q.rms', 'scenario_Q.rms',
    'scenario_Q.D_3x.event', 'calm_anchor.rms', 'calm_residual.rms',
    'exact_cap_product_scaled.rms', 'exact_repair_projection.rms',
]
rows = []
for name, path in paths.items():
    if not path.exists():
        continue
    with path.open('r', encoding='utf-8') as fh:
        data = json.load(fh)
    rows.append({'policy': name, **{key: data.get(key) for key in keys}})
pd.DataFrame(rows).set_index('policy')
